In [1]:
import numpy as np
import time
from dataclasses import dataclass
from typing import Callable, Optional
import matplotlib.pyplot as plt

In [2]:
from simulator import generate_hierarchical
from posteriors import *
from samplers_hierarchical import *

In [3]:
rng = np.random.default_rng(221)

print("Generating data")
ds = generate_hierarchical(rng=rng, truth_model = "power_law")
K = len(ds.seasons)
print(f"K = {K}, "
     f"True means = {ds.phi_k}")


Generating data
K = 10, True means = [37.0740292  27.82323994 24.08274659 26.33040153 40.39555597 30.25958186
 32.61164236 39.91307355 19.05233462 33.71709834]


In [4]:
param_names = []

for i in range(K):
    param_names.append(f"log_phi_{i+1}")

param_names.extend(["gamma", "log_eta", "delta", "mu_phi", "log_sigma_phi"])
param_names

['log_phi_1',
 'log_phi_2',
 'log_phi_3',
 'log_phi_4',
 'log_phi_5',
 'log_phi_6',
 'log_phi_7',
 'log_phi_8',
 'log_phi_9',
 'log_phi_10',
 'gamma',
 'log_eta',
 'delta',
 'mu_phi',
 'log_sigma_phi']

In [5]:
ds.true_params

{'mu_phi': np.float64(3.4011973816621555),
 'sigma_phi': 0.3,
 'gamma': 2.5,
 'eta': 500.0,
 'delta': 3.7,
 'truth_model': 'power_law',
 'K': 10}

In [6]:
theta_true = np.log(ds.phi_k).tolist()
theta_true.extend([ds.true_params["gamma"], np.log(ds.true_params["eta"]),
                   ds.true_params["delta"], ds.true_params["mu_phi"], np.log(ds.true_params["sigma_phi"])])
theta_true = np.array(theta_true)
print(f"True params, (transformed) are {theta_true}")

true_values = {
    **{f"log_phi_{k+1}": np.log(ds.phi_k[k]) for k in range(K)},
    "gamma": ds.true_params["gamma"],
    "log_eta": np.log(ds.true_params["eta"]),
    "delta": ds.true_params["delta"],
    "mu_phi": ds.true_params["mu_phi"],
    "log_sigma_phi": np.log(ds.true_params["sigma_phi"]),
}


True params, (transformed) are [ 3.6129167   3.32587164  3.18149567  3.27072422  3.69871978  3.40981289
  3.48466935  3.68670393  2.94718965  3.51800508  2.5         6.2146081
  3.7         3.40119738 -1.2039728 ]


In [7]:
prior_type = "lognormal_gamma"
parameterization = "centered"

def log_post(theta):
    return log_posterior_hierarchical(theta, ds.seasons, parameterization, prior_type)

def grad_log_post(theta):
    return grad_log_posterior_hierarchical(theta, ds.seasons, parameterization, prior_type)

In [8]:
theta_init = theta_true + rng.normal(0, 0.1, size=K+5)

In [9]:
rwmh_results = run_multiple_chains(
        run_rwmh,
        theta_init=theta_init,
        n_chains=4,
        init_strategy="jitter",
        init_scale=0.001,
        rng=rng,
        log_posterior_fn=log_post,
        n_iterations=100000,
        n_burnin=20000,
        adapt_proposal=True,
        param_names=param_names,
    )

Iteration 5000/100000: accept rate = 0.176, scale = 0.119, elapsed = 1.1s
Iteration 10000/100000: accept rate = 0.217, scale = 0.119, elapsed = 2.0s
Iteration 15000/100000: accept rate = 0.229, scale = 0.119, elapsed = 3.0s
Iteration 20000/100000: accept rate = 0.234, scale = 0.119, elapsed = 4.0s
Iteration 25000/100000: accept rate = 0.238, scale = 0.119, elapsed = 5.0s
Iteration 30000/100000: accept rate = 0.239, scale = 0.119, elapsed = 6.0s
Iteration 35000/100000: accept rate = 0.240, scale = 0.119, elapsed = 7.0s
Iteration 40000/100000: accept rate = 0.240, scale = 0.119, elapsed = 8.0s
Iteration 45000/100000: accept rate = 0.241, scale = 0.119, elapsed = 9.1s
Iteration 50000/100000: accept rate = 0.241, scale = 0.119, elapsed = 10.1s
Iteration 55000/100000: accept rate = 0.240, scale = 0.119, elapsed = 11.1s
Iteration 60000/100000: accept rate = 0.240, scale = 0.119, elapsed = 12.1s
Iteration 65000/100000: accept rate = 0.241, scale = 0.119, elapsed = 13.1s
Iteration 70000/100000

In [10]:
rwmh_cov = estimate_dense_precond_from_rwmh(rwmh_results, ridge=1e-6)

In [11]:
mala_results = run_multiple_chains(
        run_mala,
        theta_init=theta_init,
        n_chains=4,
        init_strategy="jitter",
        init_scale=0.001,
        rng=rng,
        log_posterior_fn=log_post,
        grad_log_posterior_fn=grad_log_post,
        n_iterations=100000,
        n_burnin=20000,
        step_size=1e-3,
        adapt_step=True,
        adapt_until=20000,
        target_accept=0.57,
        param_names=param_names,
        precond=rwmh_cov,
        adapt_precond=False,
        precond_type="dense",
        normalize_precond=True,
    )

Iteration 5000/100000: accept rate = 0.003, step_size = 4.72237e-06, elapsed = 1.7s
Iteration 10000/100000: accept rate = 0.272, step_size = 3.47213e-06, elapsed = 3.5s
Iteration 15000/100000: accept rate = 0.372, step_size = 4.24662e-06, elapsed = 5.3s
Iteration 20000/100000: accept rate = 0.423, step_size = 3.75714e-06, elapsed = 7.2s
Iteration 25000/100000: accept rate = 0.461, step_size = 3.75714e-06, elapsed = 9.0s
Iteration 30000/100000: accept rate = 0.487, step_size = 3.75714e-06, elapsed = 10.9s
Iteration 35000/100000: accept rate = 0.506, step_size = 3.75714e-06, elapsed = 12.7s
Iteration 40000/100000: accept rate = 0.518, step_size = 3.75714e-06, elapsed = 14.5s
Iteration 45000/100000: accept rate = 0.529, step_size = 3.75714e-06, elapsed = 16.4s
Iteration 50000/100000: accept rate = 0.537, step_size = 3.75714e-06, elapsed = 18.2s
Iteration 55000/100000: accept rate = 0.546, step_size = 3.75714e-06, elapsed = 20.0s
Iteration 60000/100000: accept rate = 0.552, step_size = 3.7

In [12]:
print_diagnostics_multi_hierarchical(
    {"RWMH": rwmh_results, "MALA": mala_results},
    parameterization="centered",
    K=10,
    latent_display="log_phi",
    true_values=true_values,
)

save_traceplots_multi_hierarchical(
    rwmh_results,
    "traceplots_hierarchical_rwmh_centered.png",
    parameterization="centered",
    K=10,
    latent_display="log_phi",
)

save_traceplots_multi_hierarchical(
    mala_results,
    "traceplots_hierarchical_mala_centered.png",
    parameterization="centered",
    K=10,
    latent_display="log_phi",
)


Sampler      Chains  Accept%  Time(s)ESS(log_phi_1)ESS(log_phi_2)ESS(log_phi_3)ESS(log_phi_4)ESS(log_phi_5)ESS(log_phi_6)ESS(log_phi_7)ESS(log_phi_8)ESS(log_phi_9)ESS(log_phi_10)ESS(gamma)ESS(log_eta)ESS(delta)ESS(mu_phi)ESS(log_sigma_phi)
------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
RWMH              4    0.255     79.4        276        199        255        284        273        242        315        292        308        240        318        999       1437        383        257
MALA              4    0.565    147.5        170        171        171        170        171        170        171        171        170        171        167        162        162        173        163

Sampler      Chains  Accept%  Time(s)Rhat(log_phi_1)Rhat(log_phi_2)Rhat(log_phi_3)Rhat(log_phi_4)Rhat(log_phi_5)Rhat(log_phi_6)Rhat(log